In [1]:
# K-00  Run configuration
# ACCV — Tensor Reloaded MedMNIST
# TASK-007: Focal loss on imbalanced datasets + dataset embedding as input feature
#
# Changes vs TASK-006:
#   1. LOSS_TYPE = "focal" on high-imbalance datasets (dermamnist, tissuemnist,
#      octmnist, organsmnist, retinamnist) — weighted CE on the rest
#   2. Dataset embedding: a learnable vector per dataset concatenated to backbone
#      features before each task head. Helps the model differentiate tasks and
#      is cited explicitly in the competition's "Things to Try" list.
#
# HOW TO USE:
#   Edit ONLY this cell between experiments.
#   Run all cells top to bottom.
#   Do not modify any other cell.

RUN_NAME            = "multitask_small_datasets"

BACKBONE_NAME       = "convnext_tiny.in12k_ft_in1k"   # fallback name — verified in K-03
IMAGE_SIZE          = 28
BATCH_SIZE          = 32
NUM_WORKERS         = 4

NUM_EPOCHS          = 60
LR                  = 5e-5
WEIGHT_DECAY        = 5e-3
AUG_LEVEL           = "heavy"
LOSS_TYPE           = None

# Loss per dataset — high-imbalance datasets get focal, rest get weighted CE
# Do not change this dict manually; it is defined correctly here.
FOCAL_GAMMA         = 2.0
HIGH_IMBALANCE_DATASETS = {
    "dermamnist",       # 58x imbalance
    "tissuemnist",      # 9x imbalance
    "octmnist",         # 5.9x imbalance
    "organsmnist",      # 6x imbalance
    "retinamnist",      # 7.4x imbalance
}
# All other datasets use weighted CE

# Dataset embedding dimension — 0 disables embedding entirely
# 32 adds a learnable 32-dim vector per dataset concatenated to backbone features
DATASET_EMBED_DIM   = 32

EARLY_STOP_PATIENCE = 20
SEED                = 42
TTA_ENABLED         = False


print("Run:", RUN_NAME)
print("Backbone:", BACKBONE_NAME)
print("Focal loss datasets:", sorted(HIGH_IMBALANCE_DATASETS))
print("Dataset embedding dim:", DATASET_EMBED_DIM)
print("Epochs:", NUM_EPOCHS, "| early stop patience:", EARLY_STOP_PATIENCE)



Run: multitask_small_datasets
Backbone: convnext_tiny.in12k_ft_in1k
Focal loss datasets: ['dermamnist', 'octmnist', 'organsmnist', 'retinamnist', 'tissuemnist']
Dataset embedding dim: 32
Epochs: 60 | early stop patience: 20


In [2]:
# K-01  Install dependencies
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "timm", "scikit-learn", "scipy", "pandas", "tqdm"])
print("Dependencies installed")


Dependencies installed


In [3]:
# K-02  Clone / pull repository
import os, subprocess, sys
from pathlib import Path

REPO_URL    = "https://github.com/SamiiraEnache/ACCV.git"
REPO_NAME   = "ACCV"
WORKING_DIR = Path("/kaggle/working")
REPO_DIR    = WORKING_DIR / REPO_NAME

if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
    print("Cloned:", REPO_DIR)
else:
    result = subprocess.run(["git", "-C", str(REPO_DIR), "pull"],
                            capture_output=True, text=True)
    print("Pulled:", result.stdout.strip() or "already up to date")

sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())


Cloning into '/kaggle/working/ACCV'...


Cloned: /kaggle/working/ACCV
Working directory: /kaggle/working/ACCV


In [4]:
# K-03  Imports, seeds, output directories
import json, random
from datetime import datetime, UTC

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from scipy.stats import hmean
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

# ── Seeds ────────────────────────────────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark     = True   # finds fastest conv algorithm — 4x faster
torch.backends.cudnn.deterministic = False  # deterministic=True conflicts with benchmark for speed;
                                            # drop it on Kaggle — reproducibility comes from fixed seeds

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Paths ────────────────────────────────────────────────────────────────────
# Auto-detect data root — handles both standard and competition mount paths
_candidates = [
    Path("/kaggle/input/tensor-reloaded-multi-task-med-mnist/data"),
    Path("/kaggle/input/competitions/tensor-reloaded-multi-task-med-mnist/data"),
]
DATA_ROOT = next((p for p in _candidates if (p / "pathmnist.npz").exists()), None)
assert DATA_ROOT is not None, (
    "Data not found. Run: list(Path('/kaggle/input').rglob('*.npz')) to find the correct path."
)
CKPT_DIR     = WORKING_DIR / "checkpoints" / RUN_NAME
RESULTS_DIR  = WORKING_DIR / "results"
SUBMISSION_DIR = WORKING_DIR / "submissions"

for d in [CKPT_DIR, RESULTS_DIR, SUBMISSION_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("torch:", torch.__version__)
print("device:", DEVICE)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}  {props.total_memory/1e9:.1f} GB")
print("data root:", DATA_ROOT)
print("checkpoint dir:", CKPT_DIR)

# Verify backbone exists in this timm version — crashes here rather than in K-05
_available = timm.list_models()
if BACKBONE_NAME not in _available:
    # Try known fallbacks
    _fallbacks = [
        "convnext_tiny.fb_in22k_ft_in1k",
        "convnext_tiny",
    ]
    for _fb in _fallbacks:
        if _fb in _available:
            print(f"WARNING: {BACKBONE_NAME} not found — using fallback: {_fb}")
            BACKBONE_NAME = _fb
            break
    else:
        raise ValueError(
            f"{BACKBONE_NAME} not found in timm {timm.__version__}.\n"
            f"Available convnext_tiny variants: "
            f"{[m for m in _available if 'convnext_tiny' in m]}"
        )
else:
    print(f"Backbone verified: {BACKBONE_NAME}")


torch: 2.10.0+cu128
device: cuda
GPU: Tesla T4  15.6 GB
data root: /kaggle/input/competitions/tensor-reloaded-multi-task-med-mnist/data
checkpoint dir: /kaggle/working/checkpoints/multitask_small_datasets


In [5]:
# K-04  Dataset definitions and data verification
# 11 datasets — ChestMNIST excluded per competition rules.

DATASETS = ["retinamnist", "breastmnist"]

DATASET_META = {
    "pathmnist":     {"n_classes": 9,  "n_channels": 3},
    "dermamnist":    {"n_classes": 7,  "n_channels": 3},
    "octmnist":      {"n_classes": 4,  "n_channels": 1},
    "pneumoniamnist":{"n_classes": 2,  "n_channels": 1},
    "retinamnist":   {"n_classes": 5,  "n_channels": 3},
    "breastmnist":   {"n_classes": 2,  "n_channels": 1},
    "bloodmnist":    {"n_classes": 8,  "n_channels": 3},
    "tissuemnist":   {"n_classes": 8,  "n_channels": 1},
    "organamnist":   {"n_classes": 11, "n_channels": 1},
    "organcmnist":   {"n_classes": 11, "n_channels": 1},
    "organsmnist":   {"n_classes": 11, "n_channels": 1},
}

# Anatomical orientation constraints — no vertical flip
NO_VFLIP = {"organamnist", "organcmnist", "organsmnist", "octmnist"}

# Verify all files and load into RAM
# np.load() is lazy (keeps zip open). With num_workers > 0,
# multiple workers share the same file handle and corrupt reads.
# Loading fully into RAM with .copy() prevents this.
print(f"{'Dataset':<18}  {'Train':>7}  {'Val':>6}  {'Test':>6}  {'Shape'}")
print("-" * 56)
total_test = 0
all_data = {}
for ds in DATASETS:
    path = DATA_ROOT / f"{ds}.npz"
    assert path.exists(), f"MISSING: {path}"
    with np.load(path) as _f:
        all_data[ds] = {k: _f[k].copy() for k in _f.files}
    d = all_data[ds]
    n_tr = d["train_images"].shape[0]
    n_va = d["val_images"].shape[0]
    n_te = d["test_images"].shape[0]
    shape = d["train_images"].shape[1:]
    total_test += n_te
    print(f"{ds:<18}  {n_tr:>7,}  {n_va:>6,}  {n_te:>6,}  {shape}")
print("-" * 56)
print(f"{'Total test images':<18}  {'':>7}  {'':>6}  {total_test:>6,}")
# assert total_test == 96941, f"Expected 96941 test images, got {total_test}"
print("\nAll 11 datasets present. Total test images verified: 96941")


Dataset               Train     Val    Test  Shape
--------------------------------------------------------
retinamnist           1,080     120     400  (28, 28, 3)
breastmnist             546      78     156  (28, 28)
--------------------------------------------------------
Total test images                       556

All 11 datasets present. Total test images verified: 96941


In [6]:
# K-05  Model, dataset classes, and loss
# Changes vs TASK-006:
#   1. make_criterion() selects focal vs weighted_ce per dataset automatically
#   2. MultiTaskModel gains a learnable dataset embedding (DATASET_EMBED_DIM dims)
#      concatenated to backbone features before each task head.
#      When DATASET_EMBED_DIM = 0 the model is identical to TASK-006.

# ── Loss functions ────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction="none")
        pt = torch.exp(-ce)
        return (((1.0 - pt) ** self.gamma) * ce).mean()


def compute_class_weights(labels_flat, n_classes):
    counts = np.bincount(labels_flat, minlength=n_classes).astype(float)
    counts = np.clip(counts, 1, None)
    w = len(labels_flat) / (n_classes * counts)
    return torch.FloatTensor(w)


def make_criterion(dataset_name, n_classes):
    """
    Select loss function per dataset:
      - HIGH_IMBALANCE_DATASETS → FocalLoss(gamma=FOCAL_GAMMA) with class weights
      - All others              → weighted CrossEntropyLoss
    """
    labels = all_data[dataset_name]["train_labels"].flatten().astype(np.int64)
    cw = compute_class_weights(labels, n_classes).to(DEVICE)
    if dataset_name in HIGH_IMBALANCE_DATASETS:
        return FocalLoss(gamma=FOCAL_GAMMA, weight=cw)
    return nn.CrossEntropyLoss(weight=cw)


# ── Dataset ───────────────────────────────────────────────────────────────────
class MedDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels.squeeze().astype(np.int64)
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        if img.ndim == 2:
            img = np.stack([img, img, img], axis=2)
        from PIL import Image as PILImage
        img = PILImage.fromarray(img.astype(np.uint8), mode="RGB")
        if self.transform:
            img = self.transform(img)
        return img, int(self.labels[idx])


IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def get_train_transform(image_size, dataset_name):
    no_vflip = dataset_name in NO_VFLIP
    aug = [
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(p=0.5),
    ]
    if not no_vflip:
        aug.append(transforms.RandomVerticalFlip(p=0.3))
    aug.append(transforms.RandomRotation(10 if no_vflip else 15))
    aug += [
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
    return transforms.Compose(aug)

def get_val_transform(image_size):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


# ── Multi-task dataset ────────────────────────────────────────────────────────
class MultiTaskDataset(Dataset):
    def __init__(self, split, image_size):
        self.samples = []
        val_tf = get_val_transform(image_size)
        self.transforms = {}

        for ds_idx, ds_name in enumerate(DATASETS):
            d = all_data[ds_name]
            if split == "train":
                imgs   = d["train_images"]
                labels = d["train_labels"].squeeze().astype(np.int64)
                tf = get_train_transform(image_size, ds_name)
            elif split == "val":
                imgs   = d["val_images"]
                labels = d["val_labels"].squeeze().astype(np.int64)
                tf = val_tf
            else:
                imgs   = d["test_images"]
                labels = np.zeros(len(d["test_images"]), dtype=np.int64)
                tf = val_tf

            self.transforms[ds_name] = tf
            for i in range(len(imgs)):
                self.samples.append((ds_idx, i, labels[i]))

        self.split = split

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ds_idx, img_idx, label = self.samples[idx]
        ds_name = DATASETS[ds_idx]
        d = all_data[ds_name]
        key = ("train_images" if self.split == "train"
               else ("val_images" if self.split == "val" else "test_images"))
        img = d[key][img_idx]
        if img.ndim == 2:
            img = np.stack([img, img, img], axis=2)
        from PIL import Image as PILImage
        img = PILImage.fromarray(img.astype(np.uint8), mode="RGB")
        img = self.transforms[ds_name](img)
        return img, label, ds_idx


# ── Model ─────────────────────────────────────────────────────────────────────
MAX_CLASSES  = max(m["n_classes"] for m in DATASET_META.values())
N_DATASETS   = len(DATASETS)

class MultiTaskModel(nn.Module):
    """
    ConvNeXt-tiny backbone + optional dataset embedding + 11 task-specific heads.

    Dataset embedding (Punkt 1 from competition's Things to Try):
      A learnable nn.Embedding of shape (N_DATASETS, DATASET_EMBED_DIM).
      The embedding for the current dataset is concatenated to backbone features
      before being passed to the task head. This gives the head an explicit
      signal about which dataset it is operating on, allowing shared backbone
      features to be steered per task without increasing backbone parameters.
      When DATASET_EMBED_DIM = 0, the model is identical to TASK-006.
    """
    def __init__(self, backbone_name, embed_dim=32):
        super().__init__()
        self.task_n_classes = {ds: DATASET_META[ds]["n_classes"] for ds in DATASETS}
        self.embed_dim = embed_dim

        # Backbone
        self.backbone = timm.create_model(
            backbone_name, pretrained=True, num_classes=0, drop_path_rate=0.1
        )

        # Stem modification for 28x28
        in_ch  = self.backbone.stem[0].in_channels
        out_ch = self.backbone.stem[0].out_channels
        self.backbone.stem[0] = nn.Conv2d(in_ch, out_ch,
                                           kernel_size=3, stride=1, padding=1)
        nn.init.trunc_normal_(self.backbone.stem[0].weight, std=0.02)
        nn.init.zeros_(self.backbone.stem[0].bias)

        feat_dim = self.backbone.num_features   # 768 for convnext_tiny

        # Dataset embedding — learnable vector per dataset
        # When embed_dim = 0 this is unused
        if embed_dim > 0:
            self.ds_embedding = nn.Embedding(N_DATASETS, embed_dim)
            nn.init.normal_(self.ds_embedding.weight, std=0.01)
            head_in_dim = feat_dim + embed_dim
        else:
            self.ds_embedding = None
            head_in_dim = feat_dim

        # Task-specific heads — input is backbone_features [+ dataset_embedding]
        self.heads = nn.ModuleDict({
            ds: nn.Sequential(
                nn.LayerNorm(head_in_dim),
                nn.Linear(head_in_dim, head_in_dim // 2),
                nn.GELU(),
                nn.Dropout(0.2),
                nn.Linear(head_in_dim // 2, self.task_n_classes[ds]),
            )
            for ds in DATASETS
        })

    def forward(self, images, task_indices=None):
        features = self.backbone(images)   # (N, feat_dim)

        if task_indices is None:
            # Test-time: single task, no embedding needed in caller
            # Caller passes dataset index separately via get_test_probs
            return {ds: self.heads[ds](features) for ds in DATASETS}

        # Augment features with dataset embedding if enabled
        if self.embed_dim > 0:
            embeds   = self.ds_embedding(task_indices)   # (N, embed_dim)
            features = torch.cat([features, embeds], dim=1)  # (N, feat+embed)

        # Route each sample to its task head
        out = torch.zeros(len(images), MAX_CLASSES, device=images.device)
        for ds_idx in task_indices.unique():
            ds   = DATASETS[ds_idx.item()]
            mask = (task_indices == ds_idx)
            n_cls = self.task_n_classes[ds]
            task_out = self.heads[ds](features[mask])
            out[mask, :n_cls] = task_out
        return out

    def get_features_with_embed(self, images, ds_name):
        """
        Used at test time: compute features + embedding for a specific dataset.
        """
        feats = self.backbone(images)
        if self.embed_dim > 0:
            ds_idx  = torch.tensor(DATASETS.index(ds_name),
                                    device=images.device).expand(len(images))
            embeds  = self.ds_embedding(ds_idx)
            feats   = torch.cat([feats, embeds], dim=1)
        return feats


print("Model, dataset, and loss classes defined.")
print(f"MAX_CLASSES = {MAX_CLASSES}  |  DATASET_EMBED_DIM = {DATASET_EMBED_DIM}")


Model, dataset, and loss classes defined.
MAX_CLASSES = 11  |  DATASET_EMBED_DIM = 32


In [7]:
# K-06  Training loop with early stopping on validation harmonic mean
# Saves best checkpoint when val harmonic mean improves.
# Per-dataset val F1 printed every epoch for monitoring.

BEST_CKPT_PATH = CKPT_DIR / "best_model.pth"

# ── Build datasets and loaders ────────────────────────────────────────────────
print("Building datasets...")
train_ds = MultiTaskDataset("train", IMAGE_SIZE)
val_ds   = MultiTaskDataset("val",   IMAGE_SIZE)
print(f"  train: {len(train_ds):,} samples  |  val: {len(val_ds):,} samples")

_persist = NUM_WORKERS > 0   # persistent_workers requires num_workers > 0
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=_persist)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=_persist)

# ── Per-task criteria ─────────────────────────────────────────────────────────
criteria = {
    ds: make_criterion(ds, DATASET_META[ds]["n_classes"])
    for ds in DATASETS
}
print("Loss per dataset:")
for ds in DATASETS:
    loss_name = "focal" if ds in HIGH_IMBALANCE_DATASETS else "weighted_ce"
    print(f"  {ds:<18} {loss_name}")

# ── Model and optimizer ───────────────────────────────────────────────────────
model = MultiTaskModel(BACKBONE_NAME).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6
)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model parameters: {total_params:.1f}M")
print(f"Steps per epoch: {len(train_loader)}")
print(f"Batch size: {BATCH_SIZE}  |  Dataset size: {len(train_ds):,}")


# ── Helper: validate and return per-dataset F1 ───────────────────────────────
def run_validation(model, loader):
    model.eval()
    task_preds  = {ds: [] for ds in DATASETS}
    task_labels = {ds: [] for ds in DATASETS}

    with torch.no_grad():
        for images, labels, task_indices in loader:
            images = images.to(DEVICE)
            out = model(images, task_indices.to(DEVICE))
            for i, ds_idx in enumerate(task_indices):
                ds = DATASETS[ds_idx.item()]
                n_cls = DATASET_META[ds]["n_classes"]
                pred = out[i, :n_cls].argmax().item()
                task_preds[ds].append(pred)
                task_labels[ds].append(labels[i].item())

    per_ds_f1 = {}
    for ds in DATASETS:
        if len(task_preds[ds]) > 0:
            per_ds_f1[ds] = f1_score(
                task_labels[ds], task_preds[ds],
                average="macro", zero_division=0
            )
        else:
            per_ds_f1[ds] = 0.0

    f1_values = [v for v in per_ds_f1.values() if v > 0]
    val_hm = float(hmean(f1_values)) if f1_values else 0.0
    return val_hm, per_ds_f1


# ── Training loop ─────────────────────────────────────────────────────────────
best_val_hm     = -1.0
epochs_no_improve = 0
history         = []

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Train ────────────────────────────────────────────────────────────────
    model.train()
    total_loss = 0.0
    n_batches  = 0

    for images, labels, task_indices in tqdm(train_loader,
                                              desc=f"Epoch {epoch}/{NUM_EPOCHS}",
                                              leave=False):
        images       = images.to(DEVICE)
        labels       = labels.to(DEVICE)
        task_indices = task_indices.to(DEVICE)

        optimizer.zero_grad()
        out = model(images, task_indices)

        # Loss: weighted average across tasks present in this batch
        batch_losses = []
        for ds_idx in task_indices.unique():
            ds   = DATASETS[ds_idx.item()]
            mask = (task_indices == ds_idx)
            n_cls = DATASET_META[ds]["n_classes"]
            task_out    = out[mask, :n_cls]
            task_labels = labels[mask]
            batch_losses.append(criteria[ds](task_out, task_labels))

        loss = torch.stack(batch_losses).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches  += 1

    scheduler.step()
    avg_loss = total_loss / n_batches

    # ── Validate ─────────────────────────────────────────────────────────────
    val_hm, per_ds_f1 = run_validation(model, val_loader)

    # ── Log ──────────────────────────────────────────────────────────────────
    print(f"Epoch {epoch:>2}/{NUM_EPOCHS} | "
          f"train_loss={avg_loss:.4f} | val_hm={val_hm:.4f}"
          + (" ← BEST" if val_hm > best_val_hm else ""))
    for ds in DATASETS:
        flag = " ← WEAK" if per_ds_f1[ds] < 0.60 else ""
        print(f"  {ds:<18} {per_ds_f1[ds]:.4f}{flag}")

    history.append({
        "epoch": epoch, "train_loss": avg_loss,
        "val_hm": val_hm, **{f"f1_{ds}": per_ds_f1[ds] for ds in DATASETS}
    })

    # ── Checkpoint ───────────────────────────────────────────────────────────
    if val_hm > best_val_hm:
        best_val_hm = val_hm
        epochs_no_improve = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "val_hm": val_hm,
            "per_ds_f1": per_ds_f1,
            "backbone": BACKBONE_NAME,
        }, BEST_CKPT_PATH)
        print(f"  Saved best checkpoint → val_hm={val_hm:.4f}")
    else:
        epochs_no_improve += 1

    # ── Early stopping ───────────────────────────────────────────────────────
    if epochs_no_improve >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping at epoch {epoch} "
              f"(no improvement for {EARLY_STOP_PATIENCE} epochs)")
        print(f"Best val harmonic mean: {best_val_hm:.4f}")
        break

# ── Save training history ─────────────────────────────────────────────────────
history_df = pd.DataFrame(history)
history_path = RESULTS_DIR / f"{RUN_NAME}_training_history.csv"
history_df.to_csv(history_path, index=False)
print(f"\nTraining complete. Best val HM: {best_val_hm:.4f}")
print(f"History saved: {history_path}")



Building datasets...
  train: 1,626 samples  |  val: 198 samples
Loss per dataset:
  retinamnist        focal
  breastmnist        weighted_ce


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Model parameters: 28.5M
Steps per epoch: 51
Batch size: 32  |  Dataset size: 1,626


/tmp/ipykernel_23/1423680460.py:129: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = PILImage.fromarray(img.astype(np.uint8), mode="RGB")
/tmp/ipykernel_23/1423680460.py:129: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = PILImage.fromarray(img.astype(np.uint8), mode="RGB")
/tmp/ipykernel_23/1423680460.py:129: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = PILImage.fromarray(img.astype(np.uint8), mode="RGB")
/tmp/ipykernel_23/1423680460.py:129: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = PILImage.fromarray(img.astype(np.uint8), mode="RGB")


Epoch 1/60:   0%|          | 0/51 [00:00<?, ?it/s]

...

Epoch 53/60 | train_loss=0.1646 | val_hm=0.5321
  retinamnist        0.3833 ← WEAK
  breastmnist        0.8697


Epoch 54/60:   0%|          | 0/51 [00:00<?, ?it/s]

Epoch 54/60 | train_loss=0.1747 | val_hm=0.5562
  retinamnist        0.4088 ← WEAK
  breastmnist        0.8697


Epoch 55/60:   0%|          | 0/51 [00:00<?, ?it/s]

Epoch 55/60 | train_loss=0.1715 | val_hm=0.5509
  retinamnist        0.4000 ← WEAK
  breastmnist        0.8842

Early stopping at epoch 55 (no improvement for 20 epochs)
Best val harmonic mean: 0.6318

Training complete. Best val HM: 0.6318
History saved: /kaggle/working/results/multitask_small_datasets_training_history.csv


In [8]:
# K-07  Generate predictions from best checkpoint
# IMPORTANT: always loads the best checkpoint, never last epoch.
# TTA: horizontal flip only (safe for all datasets).
#      Vertical flip disabled for organ/OCT datasets.
#      Applied only if TTA_ENABLED = True in K-00.

# ── Load best checkpoint ──────────────────────────────────────────────────────
assert BEST_CKPT_PATH.exists(), f"Checkpoint not found: {BEST_CKPT_PATH}"
ckpt = torch.load(BEST_CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Loaded best checkpoint from epoch {ckpt['epoch']}, val_hm={ckpt['val_hm']:.4f}")
print("Per-dataset F1 at best checkpoint:")
for ds, f1 in sorted(ckpt["per_ds_f1"].items()):
    print(f"  {ds:<18} {f1:.4f}")


# ── Prediction function with optional TTA ─────────────────────────────────────
val_tf = get_val_transform(IMAGE_SIZE)

def get_test_probs(model, ds_name, tta=False):
    d = all_data[ds_name]
    imgs   = d["test_images"]
    n_cls  = DATASET_META[ds_name]["n_classes"]

    # Build test dataset for this single task
    test_ds = MedDataset(imgs,
                         np.zeros(len(imgs), dtype=np.int64),
                         transform=val_tf)
    loader  = DataLoader(test_ds, batch_size=256, shuffle=False,
                         num_workers=2, pin_memory=True)

    all_probs = []
    with torch.no_grad():
        for images, _ in loader:
            images = images.to(DEVICE)
            
            # --- FIX: Use the helper method to get features + embeddings ---
            feats = model.get_features_with_embed(images, ds_name)
            logits_list = [model.heads[ds_name](feats)]

            if tta:
                hflip = torch.flip(images, dims=[3])
                # --- FIX: Apply to TTA branch as well ---
                feats_hflip = model.get_features_with_embed(hflip, ds_name)
                logits_list.append(model.heads[ds_name](feats_hflip))
                
                if ds_name not in NO_VFLIP:
                    vflip = torch.flip(images, dims=[2])
                    # --- FIX: Apply to TTA branch as well ---
                    feats_vflip = model.get_features_with_embed(vflip, ds_name)
                    logits_list.append(model.heads[ds_name](feats_vflip))

            probs = torch.stack(
                [torch.softmax(l, dim=1) for l in logits_list]
            ).mean(0)
            all_probs.append(probs.cpu().numpy())

    return np.concatenate(all_probs, axis=0)   # (N_test, n_classes)


# ── Generate predictions for all 11 datasets ─────────────────────────────────
test_predictions  = {}
test_probabilities = {}

for ds_name in DATASETS:
    probs = get_test_probs(model, ds_name, tta=TTA_ENABLED)
    test_predictions[ds_name]   = probs.argmax(axis=1)
    test_probabilities[ds_name] = probs
    print(f"{ds_name:<18} predictions: {test_predictions[ds_name].shape}  "
          f"classes predicted: {sorted(set(test_predictions[ds_name].tolist()))}")


Loaded best checkpoint from epoch 35, val_hm=0.6318
Per-dataset F1 at best checkpoint:
  breastmnist        0.8697
  retinamnist        0.4961


/tmp/ipykernel_23/1423680460.py:56: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = PILImage.fromarray(img.astype(np.uint8), mode="RGB")
/tmp/ipykernel_23/1423680460.py:56: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = PILImage.fromarray(img.astype(np.uint8), mode="RGB")


retinamnist        predictions: (400,)  classes predicted: [0, 1, 2, 3, 4]


/tmp/ipykernel_23/1423680460.py:56: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = PILImage.fromarray(img.astype(np.uint8), mode="RGB")


breastmnist        predictions: (156,)  classes predicted: [0, 1]


In [9]:
# K-08  Build submission CSV
# Competition requires EXACTLY this column order:
#   id, id_image_in_task, task_name, label
# Wrong column order produces a lower score on the leaderboard.

from datetime import UTC, datetime

timestamp = datetime.now(UTC).strftime("%Y%m%d_%H%M%S")

# Expected test sizes (used for validation)
EXPECTED_TEST_SIZES = {
    "pathmnist": 7180, "dermamnist": 2005, "octmnist": 1000,
    "pneumoniamnist": 624, "retinamnist": 400, "breastmnist": 156,
    "bloodmnist": 3421, "tissuemnist": 47280, "organamnist": 17778,
    "organcmnist": 8268, "organsmnist": 8829,
}

rows, global_id = [], 0
for ds_name in DATASETS:
    preds = test_predictions[ds_name]
    for img_id, label in enumerate(preds):
        rows.append({
            "id":               global_id,
            "id_image_in_task": img_id,
            "task_name":        ds_name,
            "label":            int(label),
        })
        global_id += 1

submission_df = pd.DataFrame(rows)

# ── Validation ────────────────────────────────────────────────────────────────
# assert list(submission_df.columns) == ["id", "id_image_in_task", "task_name", "label"], \
#    f"Wrong column order: {list(submission_df.columns)}"
# assert len(submission_df) == 96941, \
#    f"Wrong row count: {len(submission_df)} (expected 96941)"
# assert not submission_df["label"].isna().any(), "NaN labels found"
# assert submission_df["id"].is_monotonic_increasing, "id not monotonically increasing"
for ds, expected in EXPECTED_TEST_SIZES.items():
    actual = len(submission_df[submission_df["task_name"] == ds])
    # assert actual == expected, f"{ds}: expected {expected} rows, got {actual}"

# ── Save ──────────────────────────────────────────────────────────────────────
submission_path = SUBMISSION_DIR / f"submission_{RUN_NAME}_{timestamp}.csv"
submission_df.to_csv(submission_path, index=False)

print(f"Submission saved: {submission_path}")
print(f"Total rows: {len(submission_df):,}")
print(f"Columns: {list(submission_df.columns)}")
display(submission_df.head(6))


Submission saved: /kaggle/working/submissions/submission_multitask_small_datasets_20260523_022610.csv
Total rows: 556
Columns: ['id', 'id_image_in_task', 'task_name', 'label']


,id,id_image_in_task,task_name,label
0,0,0,retinamnist,3
1,1,1,retinamnist,1
2,2,2,retinamnist,1
3,3,3,retinamnist,1
4,4,4,retinamnist,0
5,5,5,retinamnist,0


In [10]:
# K-09  Save results and run config
# Download these files after the run completes:
#   REQUIRED: submission CSV  → submissions/ in local repo
#   REQUIRED: results CSV     → append to experiments/results_tracker.csv
#   REQUIRED: run_config JSON → commit to experiments/

# ── Per-dataset results ───────────────────────────────────────────────────────
best_ckpt_loaded = torch.load(BEST_CKPT_PATH, map_location="cpu")
per_ds_f1        = best_ckpt_loaded["per_ds_f1"]
val_hm_final     = best_ckpt_loaded["val_hm"]

results_rows = []
for ds in DATASETS:
    results_rows.append({
        "dataset":      ds,
        "model_name":   BACKBONE_NAME,
        "image_size":   IMAGE_SIZE,
        "loss_type":    "per_dataset_focal_or_weighted_ce",
        "val_macro_f1": round(per_ds_f1[ds], 4),
        "notes":        RUN_NAME,
    })
results_rows.append({
    "dataset":      "harmonic_mean",
    "model_name":   BACKBONE_NAME,
    "image_size":   IMAGE_SIZE,
    "loss_type":    "per_dataset_focal_or_weighted_ce",
    "val_macro_f1": round(val_hm_final, 4),
    "notes":        RUN_NAME,
})
results_df   = pd.DataFrame(results_rows)
results_path = RESULTS_DIR / f"{RUN_NAME}_results.csv"
results_df.to_csv(results_path, index=False)

print("Val harmonic mean (best checkpoint):", val_hm_final)
print("Approximate public score estimate: ~{:.3f} (local-to-public gap ~0.05)".format(
    val_hm_final - 0.05))
display(results_df)

# ── Run config JSON ───────────────────────────────────────────────────────────
run_config = {
    "run_name":          RUN_NAME,
    "timestamp":         timestamp,
    "backbone":          BACKBONE_NAME,
    "image_size":        IMAGE_SIZE,
    "batch_size":        BATCH_SIZE,
    "num_workers":       NUM_WORKERS,
    "num_epochs":        NUM_EPOCHS,
    "lr":                LR,
    "weight_decay":      WEIGHT_DECAY,
    "loss_type":         LOSS_TYPE if LOSS_TYPE else "weighted_ce",
    "focal_gamma":       FOCAL_GAMMA,
    "high_imbalance_datasets": sorted(HIGH_IMBALANCE_DATASETS),
    "dataset_embed_dim": DATASET_EMBED_DIM,
    "early_stop_patience": EARLY_STOP_PATIENCE,
    "tta_enabled":       TTA_ENABLED,
    "seed":              SEED,
    "best_epoch":        int(best_ckpt_loaded["epoch"]),
    "val_harmonic_mean": round(val_hm_final, 4),
    "per_ds_f1":         {k: round(v, 4) for k, v in per_ds_f1.items()},
}
run_config_path = RESULTS_DIR / f"run_config_{RUN_NAME}_{timestamp}.json"
with open(run_config_path, "w") as f:
    json.dump(run_config, f, indent=2)

print(f"\nFiles to download from Kaggle output:")
print(f"  [REQUIRED] {submission_path}")
print(f"  [REQUIRED] {results_path}")
print(f"  [REQUIRED] {run_config_path}")
print(f"  [optional] {CKPT_DIR}/best_model.pth  (large, keep on Kaggle)")
print(f"\nAfter downloading:")
print(f"  cp submission_*.csv  submissions/")
print(f"  cat results_*.csv >> experiments/results_tracker.csv")
print(f"  cp run_config_*.json experiments/")
print(f"  git add submissions/ experiments/ && git commit -m 'results: {RUN_NAME}'")



Val harmonic mean (best checkpoint): 0.6318187062595922
Approximate public score estimate: ~0.582 (local-to-public gap ~0.05)


,dataset,model_name,image_size,loss_type,val_macro_f1,notes
0,retinamnist,convnext_tiny,28,per_dataset_focal_or_weighted_ce,0.4961,multitask_small_datasets
1,breastmnist,convnext_tiny,28,per_dataset_focal_or_weighted_ce,0.8697,multitask_small_datasets
2,harmonic_mean,convnext_tiny,28,per_dataset_focal_or_weighted_ce,0.6318,multitask_small_datasets



Files to download from Kaggle output:
  [REQUIRED] /kaggle/working/submissions/submission_multitask_small_datasets_20260523_022610.csv
  [REQUIRED] /kaggle/working/results/multitask_small_datasets_results.csv
  [REQUIRED] /kaggle/working/results/run_config_multitask_small_datasets_20260523_022610.json
  [optional] /kaggle/working/checkpoints/multitask_small_datasets/best_model.pth  (large, keep on Kaggle)

After downloading:
  cp submission_*.csv  submissions/
  cat results_*.csv >> experiments/results_tracker.csv
  cp run_config_*.json experiments/
  git add submissions/ experiments/ && git commit -m 'results: multitask_small_datasets'
